In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
from sklearn.utils.class_weight import compute_class_weight
import os
import zipfile

PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')

df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']]),
})
print(f"Dữ liệu sẵn sàng! Train: {len(df_train)} | Valid: {len(df_valid)}")

✅ Dữ liệu sẵn sàng! Train: 54626 | Valid: 7310 | Test: 7286


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 
    )

print("Đang Tokenize bằng AraBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)

# Format PyTorch Tensors
tokenized_datasets["train"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_datasets["valid"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print("Hoàn tất Tokenize!")

⏳ Đang Tokenize bằng AraBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

Map:   0%|          | 0/7286 [00:00<?, ? examples/s]

✅ Hoàn tất Tokenize!


In [ ]:
print("⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...")
y_train = df_train['label'].values

# 1. Compute class weights using sklearn's compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(19), y=y_train)

# 2. Clip the class weights to avoid extreme values
cw_clipped = np.clip(cw, 0.5, 5.0)

# 3. Normalize the class weights so that they sum to 1
cw_normalized = cw_clipped / cw_clipped.mean()

# 4. Convert the normalized class weights to a PyTorch tensor
class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...


In [ ]:
print("⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO PURE EMD LOSS...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=19
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO PURE EMD LOSS...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,398,483 || all params: 137,606,438 || trainable%: 1.7430


In [ ]:
def pure_emd_loss(logits, labels, num_classes=19):
    # 1. Convert logits to probabilities using softmax
    probs = F.softmax(logits, dim=-1)
    
    # 2. Calculate the CDF of predicted probabilities 
    cdf_pred = torch.cumsum(probs, dim=-1)
    
    # 3. Convert labels to one-hot encoding and calculate the CDF of true labels 
    one_hot = F.one_hot(labels, num_classes=num_classes).float()
    cdf_true = torch.cumsum(one_hot, dim=-1)
    
    # 4. Calculate the mean squared distance between the two CDFs 
    loss_per_sample = torch.mean((cdf_pred - cdf_true) ** 2, dim=-1)
    return loss_per_sample

class PureEMDTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits 
        device = logits.device

        loss_per_sample = pure_emd_loss(logits, labels, num_classes=19)

        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels]
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()

        return (loss, outputs) if return_outputs else loss

def compute_metrics_emd(eval_pred):
    logits, labels = eval_pred
    pred_labels = np.argmax(logits, axis=-1)
    
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    acc = accuracy_score(labels, pred_labels)
    
    return {"qwk": qwk, "accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir="../saved_models/arabert_lora_pure_emd",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,
    fp16=False,  
    seed=42
)

trainer = PureEMDTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_emd,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights_tensor
)

print("BẮT ĐẦU HUẤN LUYỆN PURE EMD LOSS (100% EMD)...")
trainer.train()

trainer.save_model("../saved_models/arabert_lora_pure_emd_best")
tokenizer.save_pretrained("../saved_models/arabert_lora_pure_emd_best")

🚀 BẮT ĐẦU HUẤN LUYỆN PURE EMD LOSS (100% EMD)...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 0.0253, 'grad_norm': 0.7336209416389465, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 0.0207, 'grad_norm': 0.24198292195796967, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 0.0195, 'grad_norm': 0.33375707268714905, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.019808730110526085, 'eval_qwk': 0.7718767580846271, 'eval_accuracy': 0.40068399452804376, 'eval_runtime': 30.5927, 'eval_samples_per_second': 238.946, 'eval_steps_per_second': 14.938, 'epoch': 1.0}
{'loss': 0.0177, 'grad_norm': 0.2782774865627289, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 0.0165, 'grad_norm': 0.38453924655914307, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 0.0164, 'grad_norm': 0.2835356593132019, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.017275648191571236, 'eval_qwk': 0.8036349774743767, 'eval_accuracy': 0.4960328317373461, 'eval_runtime': 30.7951, 'eval_samples_per_second': 237.375, 'eval_steps_per_second': 14.84, 'epoch': 4.0}
{'loss': 0.0118, 'grad_norm': 0.11229369789361954, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.0112, 'grad_norm': 0.190341517329216, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.0111, 'grad_norm': 0.2519264817237854, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.0107, 'grad_norm': 0.26477766036987305, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.017795534804463387, 'eval_qwk': 0.8003539285198042, 'eval_accuracy': 0.5002735978112175, 'eval_runtime': 30.3363, 'eval_samples_per_second': 240.966, 'eval_steps_per_second': 15.064, 'epoch': 5.0}
{'train_runtime': 3562.4083, 'train_samples_per_second': 76.67, 'train_steps_per_second': 2.396, 'train_loss': 0.015013605579690201, 'epoch': 5.0}


('../saved_models/arabert_lora_pure_emd_best\\tokenizer_config.json',
 '../saved_models/arabert_lora_pure_emd_best\\special_tokens_map.json',
 '../saved_models/arabert_lora_pure_emd_best\\vocab.txt',
 '../saved_models/arabert_lora_pure_emd_best\\added_tokens.json',
 '../saved_models/arabert_lora_pure_emd_best\\tokenizer.json')

In [ ]:
print("ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (PURE EMD)...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions  # Kích thước [7310, 19]
true_labels = predictions_output.label_ids.astype(int)

final_pred_labels = np.argmax(logits, axis=-1)

print("=== BÁO CÁO F1-SCORE PURE EMD LOSS ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

print("=== CÁC CHỈ SỐ METRIC BAREC ===")
print(f"QWK (Main Metric)       : {cohen_kappa_score(true_labels, final_pred_labels, weights='quadratic'):.4f}")
print(f"Acc19 (Exact Match)     : {accuracy_score(true_labels, final_pred_labels):.4f}")
print(f"Adjacent Acc (±1 Level)  : {np.mean(np.abs(true_labels - final_pred_labels) <= 1):.4f}")
print(f"Avg Distance (MAE)      : {mean_absolute_error(true_labels, final_pred_labels):.4f}")

🔍 ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (PURE EMD)...


  0%|          | 0/457 [00:00<?, ?it/s]


📊 === BÁO CÁO F1-SCORE PURE EMD LOSS ===
              precision    recall  f1-score   support

     Level_1       0.78      0.73      0.75        44
     Level_2       0.47      0.65      0.54        68
     Level_3       0.49      0.67      0.56       182
     Level_4       0.23      0.72      0.34        78
     Level_5       0.49      0.50      0.49       417
     Level_6       0.30      0.56      0.39       189
     Level_7       0.57      0.55      0.56       701
     Level_8       0.68      0.57      0.62       613
     Level_9       0.38      0.62      0.48       236
    Level_10       0.72      0.69      0.71      1012
    Level_11       0.28      0.30      0.29       409
    Level_12       0.50      0.38      0.43      1491
    Level_13       0.46      0.35      0.40       349
    Level_14       0.57      0.51      0.54      1072
    Level_15       0.21      0.15      0.17       258
    Level_16       0.18      0.39      0.24       114
    Level_17       0.29      0.41      